# Preprocesado

*Mantenimiento Predictivo para Aerogeneradores Offshore con Restricciones Ambientales. Trabajo de Fin de Máster. Segundo de los ocho notebooks.*

El notebook anterior nos mostró dos parques que comparten formato y protocolo y no se parecen en casi nada más. Este ejecuta el procedimiento que los prepara para los modelos: diez etapas, cada una activable y configurable por parque.

> **Aviso: este cuaderno ESCRIBE en `data/processed/`.** Las celdas 6 y 8 reescriben los doce
> ficheros del corpus canónico sin preguntar, porque `_save` no comprueba si
> ya existen. Por eso la primera celda de código lleva una **guarda**: si el corpus ya está,
> aborta antes de calcular nada y dice cómo forzarlo.

## 1. Preparación del entorno

In [1]:
import os
import sys
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Solo detiene la ejecucion si `allow_errors` es False (run_nb.py)
_proc = project_root / 'data' / 'processed'
_nombres = [f'farm_{_f}_{_s}' for _f in 'bc' for _s in
            ('train_windows.npz', 'test_sequences.npz', 'test_metadata.json',
             'col_means.npy', 'feature_cols.txt', 'preprocessing_report.json')]
_ya = [_n for _n in _nombres if (_proc / _n).exists()]
if _ya and os.environ.get('REGENERAR_CORPUS') != '1':
    raise SystemExit(f'El corpus canonico ya existe ({len(_ya)} de 12 ficheros en {_proc}) y este '
                     'cuaderno lo reescribiria. No se ha calculado nada. Forzarlo: REGENERAR_CORPUS=1')

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s', datefmt='%H:%M:%S')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

from src.preprocessing import load_config, PreprocessingPipeline

## 2. La configuración

`load_config()` lee `config/preprocessing_config.yaml`, la configuración base. Ninguno de los dos parques se preprocesa con ella directamente, cada uno entra con su configuración de `config/experiments/`.

| Parque | `data.dedup_train` | `smoothing.enabled` | Configuracion     |
|---|---|---|-------------------|
| **B** | `true` | `false` | `l6_light.yaml`   |
| **C** | ausente, `false` por defecto | `true` | `canonico_C.yaml` |

In [2]:
import yaml


def config_de(parche: str) -> dict:
    """Configuración base fusionada con un parche de `config/experiments/`"""
    def merge(base: dict, patch: dict) -> dict:
        result = base.copy()
        for k, v in patch.items():
            if k in result and isinstance(result[k], dict) and isinstance(v, dict):
                result[k] = merge(result[k], v)
            else:
                result[k] = v
        return result

    with open(project_root / 'config' / 'experiments' / parche, encoding='utf-8') as f:
        return merge(load_config(), yaml.safe_load(f))

config = load_config()

## 3. Las diez etapas sobre el parque B

La ejecución imprime cada etapa con su recuento. **Conviene tener en cuenta que la numeración de la salida y la de la tabla de etapas de la memoria no son la misma**, porque la salida agrupa tres etapas en una y añade dos pasos que la memoria trata como decisiones transversales:

| Notebook                               | Etapa en la memoria                                                                                                                                                   |
|----------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `[1/10] Loading data`                  | **1.** Carga y partición                                                                                                                                              |
| `[2/10] Zero-as-missing handling`      | **2.** Ceros espurios                                                                                                                                                 |
| `[3/10] Feature selection`             | **3.** Selección de variables                                                                                                                                         |
| `[4/10] Outlier detection & treatment` | **4.** Atípicos                                                                                                                                                       |
| `[5/10] Temperature normalisation`     | **5.** Temperatura relativa                                                                                                                                           |
| `[6/10] Savitzky-Golay smoothing`      | **6.** Suavizado — en B la configuracion lo apaga, y la etapa lo dice: `Smoother: disabled — skipping`                                                                |
| `[7/10] Building training data`        | **7.** Filtro de estado · **8.** Curva de potencia · **9.** Transiciones, y, por delante de las tres, la deduplicación, que **no aparece numerada** y se anota aparte |
| `[8/10] Temporal windowing`            | **10.** Ventaneo                                                                                                                                                      |
| `[9/10] NaN imputation`                | decisión transversal: los ausentes residuales se imputan con medias calculadas únicamente sobre el entrenamiento                                                      |
| `[10/10] Saving outputs`               | escritura de los artefactos en `data/processed/`                                                                                                                      |

Tres cosas que mirar en la salida:

- **La selección de variables deja B en 30**, de 77 candidatas, tras podar las constantes y las correlacionadas por encima de 0.80.
- **La deduplicación es la etapa que más retira**: 786 937 a 509 170 filas. No lleva número propio, así que su efecto se lee en la línea siguiente: el filtro de estado ya parte de 509 170, no de 786 937.
- **El filtro de estado** deja 509 170 en 454 596 filas, quedándose solo con operación normal, y **la exclusión de transiciones** retira 5 417 filas repartidas en los 6 eventos de fallo, 168 h antes del inicio y 72 h después del fin de cada uno.

In [3]:
pipeline_b = PreprocessingPipeline(config_de('l6_light.yaml'))
result_b = pipeline_b.run('B')

print(f'  train_windows : {result_b.train_windows.shape}')
print(f'  test_sequences: {len(result_b.test_sequences)}')
print(f'  features      : {len(result_b.feature_cols)}')

16:01:54 | INFO     | src.preprocessing.pipeline |   PIPELINE START — Farm B


16:01:54 | INFO     | src.preprocessing.pipeline |   PIPELINE START — Farm B


16:01:54 | INFO     | src.preprocessing.pipeline | [1/10] Loading data …


16:01:54 | INFO     | src.preprocessing.pipeline | [1/10] Loading data …


16:01:54 | INFO     | src.preprocessing.data_loader | Loading Farm B from .\data\raw\CARE_To_Compare\Wind Farm B


16:01:54 | INFO     | src.preprocessing.data_loader | Loading Farm B from .\data\raw\CARE_To_Compare\Wind Farm B


... (101 outputs left out)

16:02:55 | INFO     | src.preprocessing.pipeline | Report saved: .\data\processed\farm_b_preprocessing_report.json


16:02:55 | INFO     | src.preprocessing.pipeline | PIPELINE COMPLETE — Farm B | 411,249 train windows | 15 test sequences


16:02:55 | INFO     | src.preprocessing.pipeline | PIPELINE COMPLETE — Farm B | 411,249 train windows | 15 test sequences


  train_windows : (411249, 27, 30)
  test_sequences: 15
  features      : 30


## 4. El mismo recorrido sobre el parque C

Mismas diez etapas con su configuración, **2 358 533 ventanas de entrenamiento frente a las 411 249 de B**, y 35 variables seleccionadas de 280 candidatas frente a las 30 de 77.

Las proporciones, en cambio, se parecen mucho más de lo que sugieren los totales, el filtro de estado conserva el 89 % de las filas en los dos parques. Lo que cambia es el volumen y el reparto de lo que se descarta, no la lógica.

In [4]:
pipeline_c = PreprocessingPipeline(config_de('canonico_C.yaml'))
result_c = pipeline_c.run('C')

print(f'  train_windows : {result_c.train_windows.shape}')
print(f'  test_sequences: {len(result_c.test_sequences)}')
print(f'  features      : {len(result_c.feature_cols)}')

16:02:55 | INFO     | src.preprocessing.pipeline |   PIPELINE START — Farm C


16:02:55 | INFO     | src.preprocessing.pipeline |   PIPELINE START — Farm C


16:02:55 | INFO     | src.preprocessing.pipeline | [1/10] Loading data …


16:02:55 | INFO     | src.preprocessing.pipeline | [1/10] Loading data …


16:02:55 | INFO     | src.preprocessing.data_loader | Loading Farm C from .\data\raw\CARE_To_Compare\Wind Farm C


16:02:55 | INFO     | src.preprocessing.data_loader | Loading Farm C from .\data\raw\CARE_To_Compare\Wind Farm C


... (227 outputs left out)

16:31:52 | INFO     | src.preprocessing.pipeline | Report saved: .\data\processed\farm_c_preprocessing_report.json


16:31:52 | INFO     | src.preprocessing.pipeline | PIPELINE COMPLETE — Farm C | 2,358,533 train windows | 58 test sequences


16:31:52 | INFO     | src.preprocessing.pipeline | PIPELINE COMPLETE — Farm C | 2,358,533 train windows | 58 test sequences


  train_windows : (2358533, 27, 35)
  test_sequences: 58
  features      : 35


## 5. Resumen de los dos recorridos

La tabla reúne lo que cada parque produce: ventanas de entrenamiento, secuencias de test y número de variables.

In [5]:
summary = pd.DataFrame([
    {
        'Farm': 'B',
        'Train windows': result_b.train_windows.shape[0],
        'Test sequences': len(result_b.test_sequences),
        'Features': len(result_b.feature_cols),
        'Window shape': str(list(result_b.train_windows.shape[1:]))
    },
    {
        'Farm': 'C',
        'Train windows': result_c.train_windows.shape[0],
        'Test sequences': len(result_c.test_sequences),
        'Features': len(result_c.feature_cols),
        'Window shape': str(list(result_c.train_windows.shape[1:]))
    }
])
display(summary)
print(f'Outputs saved to {config["data"]["processed_path"]}/')

,Farm,Train windows,Test sequences,Features,Window shape
0,B,411249,15,30,"[27, 30]"
1,C,2358533,58,35,"[27, 35]"


Outputs saved to data/processed/



Los recuentos de cada etapa, tal como los imprime la ejecución de este notebook:

| Etapa | **Parque B**                                | **Parque C**                        |
|---|---------------------------------------------|-------------------------------------|
| Filas cargadas | 859 065 (9 turbinas, 15 eventos)            | 3 187 136 (22 turbinas, 58 eventos) |
| Selección de variables | 77 a **30**                                 | 280 a **35**                        |
| Deduplicación del entrenamiento | 786 937 a 509 170                           | no se aplica                        |
| Filtro de estado (solo operación normal) | 509 170 a 454 596                           | 3 028 608 a 2 703 901               |
| Curva de potencia | 87 filas retiradas                          | 14 862 filas retiradas              |
| Exclusión de transiciones (168 h / 72 h) | 5 417 filas, 6 eventos                      | 51 362 filas, 27 eventos            |
| Ventaneo de entrenamiento | **411 249** ventanas de 27 × 30             | **2 358 533** ventanas de 27 × 35   |
| Secuencias de test | 15 (6 anomalía / 9 normal), 70 540 ventanas | 58 (27 / 31), 149 270 ventanas      |
| Imputación de residuales | 1 809 129 valores                           | 34 167 342 valores                  |

**La divergencia entre parques que dejó el análisis exploratorio se confirma en el tratamiento.** No es solo que C sea mayor, sino que las dos rutas piden configuraciones opuestas, de modo que **una sola invocación no puede producir las dos limpiezas**. Ese es el motivo de que el preprocesado se lance una vez por parque y con su propia configuración, y de que este notebook construya un `PreprocessingPipeline` por parque en lugar de compartir uno.

Cuatro decisiones del preprocesado:

- **El escalado no forma parte del preprocesado**, sino del modelo, y se ajusta solo sobre entrenamiento para que nada del test contamine la preparación. La etapa `[9/10]` lo dice explícitamente.
- **El modelo aprende solo de operación normal.** La cadena que lo garantiza (deduplicación, filtro de estado, curva de potencia y exclusión de transiciones) se aplica únicamente sobre el tramo de entrenamiento.
- **Los ausentes residuales que el ventaneo admite se imputan con medias del entrenamiento**, nunca del conjunto completo.
- **La ventana es de 27 muestras (4.5 h) con paso 1**, etiquetada por su punto medio, y solo se construye dentro de segmentos temporalmente continuos con al menos un 90 % de valores válidos.